# BigQuery: Parameterized Queries, Dataset & Table Insights, Undelete Dataset Demo

This notebook demonstrates several powerful BigQuery features:
1. **Parameterized Queries (GA)**: How to write flexible queries that use [parameters](https://docs.cloud.google.com/bigquery/docs/parameterized-queries?_gl=1*8kkolg*_ga*MTEwNTg0MDcwNS4xNzcyNTYxNTkx*_ga_WH2QY8WWF5*czE3NzMzNzg1MjAkbzE1JGcxJHQxNzczMzc5MTI2JGo1NiRsMCRoMA..) to prevent SQL injection and improve reusability. [Release Detail](https://docs.cloud.google.com/bigquery/docs/release-notes#February_02_2026)
2. **Dataset Insights (Preview)**: How to generate dataset insights to understand relationship between various tables. [Release Detail](https://docs.cloud.google.com/bigquery/docs/release-notes#February_12_2026)
2. **Table Insights**: How to generate and publish [table insights](https://docs.cloud.google.com/bigquery/docs/generate-table-insights#insights-bigquery-table) and how to perform customization on table and column descriptions. [Release Detail](https://docs.cloud.google.com/bigquery/docs/release-notes#February_09_2026)
3. **Dataset Deletion and Undeletion (GA)**: How to delete a dataset and then recover it using the '[undelete](https://docs.cloud.google.com/bigquery/docs/restore-deleted-datasets)' feature. [Release Detail](https://docs.cloud.google.com/bigquery/docs/release-notes#February_23_2026)

### 1. Setup
First, we'll install the necessary libraries and authenticate with your Google Cloud account.

In [ ]:
!pip install google-cloud-bigquery
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
project_id = 'YOUR_PROJECT_ID'  # @param {type:"string"}
client = bigquery.Client(project=project_id)

### 2. Create a New Dataset
We will create a new dataset to hold our tables. We'll set the location to 'US' and give it a unique name.

In [ ]:
dataset_id = f'{project_id}.TheSales_data_2022'
dataset = bigquery.Dataset(dataset_id)
dataset.location = 'US'
dataset = client.create_dataset(dataset, exists_ok=True)
print(f'Dataset {dataset.dataset_id} created.')

### 3. Create Tables with Data from a Public Dataset

Now, we'll create three tables in our new dataset: `orders`, `users`, and `products`. We will copy data from the `bigquery-public-data.thelook_ecommerce` public dataset, but we will only copy data from the year 2022 to keep our tables smaller.

In [ ]:
queries = {
    'orders': f'''
        CREATE OR REPLACE TABLE `{dataset_id}.orders` AS
        SELECT * FROM `bigquery-public-data.thelook_ecommerce.orders`
        WHERE EXTRACT(YEAR FROM created_at) = 2022
    ''',
    'users': f'''
        CREATE OR REPLACE TABLE `{dataset_id}.users` AS
        SELECT * FROM `bigquery-public-data.thelook_ecommerce.users`
        WHERE EXTRACT(YEAR FROM created_at) = 2022
    ''',
    'products': f'''
        CREATE OR REPLACE TABLE `{dataset_id}.products` AS
        SELECT * FROM `bigquery-public-data.thelook_ecommerce.products`
    ''',
    'order_items': f'''
        CREATE OR REPLACE TABLE `{dataset_id}.order_items` AS
        SELECT * FROM `bigquery-public-data.thelook_ecommerce.order_items`
        WHERE EXTRACT(YEAR FROM created_at) = 2022
    '''
}

for table_name, sql in queries.items():
    print(f'Populating {table_name}...')
    client.query(sql).result()
    print(f'Table {table_name} created successfully.')

### 4. Parameterized Query

Using parameterized queries is a best practice to prevent SQL injection. Here, we'll query our new `orders` table for orders with a specific status. The status will be provided as a parameter.

In [ ]:
query = f'''
    SELECT
        p.name as product_name,
        COUNT(oi.id) as units_sold
    FROM `{dataset_id}.order_items` oi
    JOIN `{dataset_id}.products` p ON oi.product_id = p.id
    WHERE oi.status = @status
    GROUP BY 1
    ORDER BY 2 DESC
    LIMIT 5
'''

job_config = bigquery.QueryJobConfig(
    query_parameters=[
        bigquery.ScalarQueryParameter('status', 'STRING', 'Complete'),
    ]
)

results = client.query(query, job_config=job_config).to_dataframe()
print("Top Products for 'Complete' orders:")
print(results)

### 5. Generate Dataset Insights


Go to Bigquery Console locate dataset and click on the Dataset > Insights and then click on Generate to generate the insights.



### 6. Generate Table Insights


Go to Bigquery Console locate table and click on the table. Goto Insights tab and then click on Generate and Publish to generate the table insights.

### 7. Delete the Dataset

Now we'll delete the entire dataset, including all its tables. We will use the `delete_dataset` method with `delete_contents=True` to remove the tables within it.

In [ ]:
print(f'Deleting dataset {dataset_id}...')
#client.delete_dataset(dataset_id, delete_contents=True, not_found_ok=True)
#print('Dataset deleted.')

client.query(f"DROP SCHEMA `{dataset_id}` CASCADE").result()
print("Dataset successfully Deleted!")

### 8. Undelete the Dataset

BigQuery allows you to recover a deleted dataset. To do this via the CLI, you usually need to specify a time-travel timestamp (represented in milliseconds since epoch) from before the deletion.

In [ ]:
# Note: Replace 1672531200000 with a real timestamp from just before deletion if needed.
# Format: bq cp dataset.table@timestamp new_dataset.table
#print(f"bq cp {dataset_id}.orders@-3600000 {dataset_id}.orders_recovered")
print(f'Recovering dataset {dataset_id}...')
client.query(f"UNDROP SCHEMA `{dataset_id}`").result()
print("Dataset successfully recovered!")